# Part 1: agentic retrieval で knowledge base を構築する

Zava のアクセスバッジとノート PC を受け取ったばかりのあなた。最初の仕事は、社内の HR や福利厚生に関する質問に答えるための AI 駆動の knowledge system を構築することです。

**🎯 ミッション**
- クラウドアーキテクチャの PDF を **File Knowledge Source** に直接アップロードしてクエリする
- ビルド済みの 2 つの検索インデックス（**HR docs** と **health benefits docs**）を **Indexed Knowledge Sources** として接続する
- 各サブクエスチョンを自動的に適切なソースへルーティングする、マルチソース **Knowledge Base** を構築する


## 構成要素

Foundry IQ（Azure AI Search）における **knowledge base** とは、chat completion model と 1 つ以上の knowledge source を結びつけるトップレベルのリソースです。どのデータソースをクエリするか、どのモデルで推論するか、そしてクエリ実行をどのように最適化するかを定義します。

このノートブックでは、次の 2 種類の knowledge source を扱います。

1. **File Knowledge Source**: 生のファイルを直接アップロードします。チャンク化、埋め込み生成、インデックス作成はサービスが自動で行います。
2. **Indexed Knowledge Sources**: ビルド済みの検索インデックスに接続します。スキーマや処理を完全に制御したい本番ワークロード向けです。

## 事前構成済みの環境

このラボでは、以下の Azure リソースが事前にセットアップされています。

* Azure AI Search
  * `hrdocs` インデックス: HR ポリシー、ハンドブック、会社情報
  * `healthdocs` インデックス: 健康関連の福利厚生、保険、補償範囲
  * Semantic ranking 有効、埋め込みは事前計算済み
* Microsoft Foundry Models 経由の Azure OpenAI
  * `gpt-5.4`: 推論と回答生成のための chat completion
  * `text-embedding-3-large`: ベクトル検索用の embedding model
* Microsoft Fabric

## Step 1: 環境変数をセットアップする

Azure リソースの設定を読み込みます。プロジェクトフォルダーの `.env` ファイルには、Azure AI Search のエンドポイント、Azure OpenAI の認証情報、モデル構成など必要なものがすべて入っています。

**ℹ️ 注意**
- 下のセルを初めて実行するとき、kernel の選択を求められます。**Python Environments** を選び、**.venv** 環境を選択してください。
- 「Do you want to allow public and private networks to access this app?」というプロンプトが表示されることもあります。**Allow** を選択してください。

**⚠️ トラブルシューティング:**
コードセルがハングしてスピナーが止まらない場合は、次の手順を試してください。
1. ノートブック上部のツールバーから **Restart** を選択する。それでも解決しない場合:
2. VS Code のコマンドパレットから **Reload window** を実行する。それでも解決しない場合:
3. VS Code を完全に閉じてから起動し直す。

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


---

## A: File Knowledge Source

**File Knowledge Source** を使うと、生のファイルを直接アップロードできます。チャンク化、埋め込み生成、インデックス作成はサービスが自動で行います。knowledge base にドキュメントを入れる最も手早い方法です。

## Step 2: File Knowledge Source を作成する

embedding model を指定した file knowledge source を構成し、自動取り込みを有効化します。`ingestion_parameters` では次を指定します。
- **Embedding model**: アップロードされたコンテンツのベクトル埋め込みを生成します
- **Content extraction mode**: ドキュメントからテキストを抽出する方法

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    FileKnowledgeSource,
    FileKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "file-docs-knowledge-source"

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

file_ks = FileKnowledgeSource(
    name=FILE_KNOWLEDGE_SOURCE_NAME,
    description="Uploaded HR and company documents for Zava",
    file_parameters=FileKnowledgeSourceParameters(
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AZURE_OPENAI_ENDPOINT,
                    deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                    model_name=AZURE_OPENAI_EMBEDDING_MODEL,
                    api_key=AZURE_OPENAI_KEY,
                )
            ),
            content_extraction_mode="minimal",
            )
    ),
)

index_client.create_or_update_knowledge_source(file_ks)
print(f"File knowledge source '{FILE_KNOWLEDGE_SOURCE_NAME}' created or updated successfully.")

## Step 3: knowledge source にファイルをアップロードする

ドキュメントを file knowledge source に直接アップロードします。サービスがテキストを自動抽出し、検索可能なセグメントに分割し、ベクトル埋め込みを生成します。

⏱️ サンプルファイルの取り込みには約 40 秒かかります。

In [ ]:
from pathlib import Path

data_dir = Path("..") / "data" / "ai-search-data"
files_to_upload = [
    data_dir / "filesource" / "MSFT_cloud_architecture_zava.pdf"
]

uploaded_files = []
for file_path in files_to_upload:
    with open(file_path, "rb") as f:
        file_content = f.read()
    uploaded = index_client.upload_knowledge_source_file(
        FILE_KNOWLEDGE_SOURCE_NAME, file_content, filename=file_path.name
    )
    uploaded_files.append(uploaded)
    print(f"Uploaded: {file_path.name} (file_id: {uploaded.file_id}, {uploaded.file_size_bytes} bytes)")

print(f"\nTotal files uploaded: {len(uploaded_files)}")

## Step 4: file source 用の knowledge base を作成する

knowledge base は、次の要素を束ねるオーケストレーション層です。

1. データソース（先ほどの file knowledge source）
2. クエリの理解と回答生成のための LLM（Azure OpenAI）
3. クエリの処理方法とレスポンスのフォーマット設定

この knowledge base は出力モードに `ANSWER_SYNTHESIS` を使用しているため、アタッチした LLM が取得したチャンクから自然な日本語（または元のクエリの言語）で完結した回答を生成します。

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalOutputMode,
)

FILE_KNOWLEDGE_BASE_NAME = "file-docs-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=FILE_KNOWLEDGE_BASE_NAME,
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name=FILE_KNOWLEDGE_SOURCE_NAME),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{FILE_KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## Step 5: file knowledge base にクエリを投げる

アップロードしたドキュメントについて質問してみましょう。knowledge base はファイルの内容を横断検索し、引用付きの自然言語の回答を合成します。

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)
from IPython.display import Markdown, display

file_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=FILE_KNOWLEDGE_BASE_NAME, credential=search_credential
)

file_ks_params = FileKnowledgeSourceParams(
    knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = "As a new Zava engineer, what cloud architecture principles should I know about? What data sensitivity levels does Zava use?"

req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(text=question)
            ],
        )
    ],
    knowledge_source_params=[file_ks_params],
    include_activity=True,
)

result = file_kb_client.retrieve(retrieval_request=req)
display(Markdown(result.response[0].content[0].text))

### activity log を確認する

activity log には、knowledge base が実行した一連のステップが記録されています。各 activity には次が含まれます。

* `type`: activity の種別。今回のログでは、最初の `modelQueryPlanning`、（file knowledge source へ送られたクエリごとの）複数の `file` ステップ、回答合成ステップの `modelAnswerSynthesis`、semantic re-ranking ステップの `agenticReasoning` が表示されます。
* `elapsedMS`: その activity にかかった時間。
* `id`: activity の ID。各 reference がどの activity から生成されたかをひも付けるのに使えます。reference は次に確認します。

activity のエントリには、種別固有のメタデータも含まれます。これにより、モデル駆動のステップでどれだけトークンが使われたか、取得ステップでどのようなクエリがソースに送られたかを確認できます。

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### references を確認する

次のセルでは、knowledge base が返す生の `references` 配列を表示します。

references を見ると、どのソースから取得したか、各 reference のソースメタデータが分かります。各結果には次が含まれます。

* `type`: この knowledge base は file source のみを含むため、常に `file` です。
* `id`: reference の数値 ID。回答中に `[ref:1]` のような引用マーカーとして登場します。
* `activitySource`: この reference を生成した取得 activity の ID。activity log のどの検索ステップがこのドキュメントを返したかをたどれます。
* `sourceData`: 自動作成された検索インデックスから取得したチャンクのフィールド。file source のインデックスでは常に `uid`、`snippet`、`metadata_storage_path` が含まれます。
* `rerankerScore`: re-ranking モデルによる 0〜4 のスコア。4 が最も関連性が高いことを示します。
* `docName`: アップロードされたファイルに対して自動生成されたファイル名。

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

---

## B: Indexed Knowledge Sources

**Indexed Knowledge Sources** はビルド済みの検索インデックスに接続し、スキーマ、チャンク化、埋め込みを完全に制御できます。大規模データセットを扱う本番ワークロードに最適です。

このセクションでは、取り込み済みのドキュメントチャンクを格納した 2 つの検索インデックスを事前に用意しています。通常はインデクサーパイプラインを構成して、Azure Blob Storage などのソースから検索インデックスへドキュメントを取り込みます。

## Step 6: 検索インデックスを確認する

下のセルを実行して、各インデックスに想定どおりの数のドキュメントチャンクが入っていることを確認してください。

* `hrdocs`: HR ポリシーに関する 50 件のドキュメントチャンク
* `healthdocs`: 健康関連の福利厚生・保険に関する 334 件のドキュメントチャンク

In [ ]:
from azure.search.documents import SearchClient

for index_name in [HRDOCS_INDEX, HEALTHDOCS_INDEX]:
    client = SearchClient(AZURE_SEARCH_SERVICE_ENDPOINT, index_name, search_credential)
    count = client.get_document_count()
    print(f"{index_name}: {count} documents")


## Step 7: 2 つの indexed knowledge source を作成する

各インデックスを指す knowledge source を 2 つ作成します。

* `healthdocs-knowledge-source`: `healthdocs` インデックスを参照
* `hrdocs-knowledge-source`: `hrdocs` インデックスを参照

いずれも検索対象フィールドとして `snippet` を指定し、引用リンクにパスを使えるよう `blob_path` を source data field として含めます。

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [HR_KNOWLEDGE_SOURCE_NAME, HEALTH_KNOWLEDGE_SOURCE_NAME]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
    print(f"Knowledge source '{knowledge_source.name}' created or updated successfully.")


## Step 8: マルチソースの knowledge base を作成する

両方の search index knowledge source を含む knowledge base を作成します。前回の knowledge base と同様に `output_mode=ANSWER_SYNTHESIS` を使用し、アタッチした LLM で根拠付きの引用を伴う回答を生成します。

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-search-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base over HR and health document indexes",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## Step 9: マルチソースの knowledge base にクエリを投げる

2 つの knowledge source にまたがる、2 つの観点を含むオンボーディングの質問を投げてみましょう。

* _"How many vacation weeks does a senior Zava employee get?"_ → HR 関連なので `hrdocs` から取得されるはず
* _"Which health plan gives the best coverage for mental health services?"_ → 福利厚生関連なので `healthdocs` から取得されるはず

knowledge base は agentic retrieval を使って次を行います。

1. クエリを分析し、2 つの異なるトピックがあることを認識する
2. 焦点を絞ったサブクエリに分解する
3. 各サブクエリに対して関連する knowledge source を選択する
4. 両方のソースに対して並行に検索を実行する
5. 結果をマージして返す

In [ ]:
from azure.search.documents.knowledgebases.models import (
    SearchIndexKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

hrdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)
healthdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = (
    "I just joined Zava as a senior engineer. "
    "According to company policy, how many vacation weeks does a senior employee get? "
    "And among the available health plans, which gives the best coverage for mental health services?"
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[hrdocs_knowledge_source_params, healthdocs_knowledge_source_params],
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
display(Markdown(result.response[0].content[0].text))

### activity log を確認する

今回の activity log では、各検索インデックスへ送られたクエリごとに複数の `searchIndex` ステップが表示されます。

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### references を確認する

今回は references の `type` が `searchIndex` になり、`sourceData` のフィールド構成も少し異なります。これらの検索インデックスは、自動作成された file source のインデックスとは異なるフィールド構成になっているためです。

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

上の reference と activity log を見て、次を確認してみてください。

1. **vacation（休暇）** の質問はどの knowledge source が答えましたか？
2. **mental health plan（メンタルヘルスのプラン）** の質問はどの knowledge source が答えましたか？
3. agent は **両方の** ソースをクエリしましたか、それとも各サブクエリに対して最も関連性の高い 1 つだけをクエリしましたか？

## ボーナス: Copilot CLI からクエリする

すべての knowledge base は MCP サーバーのエンドポイントを公開しており、VS Code や GitHub Copilot CLI など MCP 互換のクライアントから追加できます。
Copilot CLI から knowledge base にクエリを投げてみたい場合は、[Copilot CLI sidequest](copilot-cli-sidequest.md) のセットアップ手順に従い、下のコマンドを実行して Zava の knowledge base を MCP サーバーとして利用してください。

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
print(f"copilot mcp add zava-kb \"{mcp_url}\" --header \"api-key={AZURE_SEARCH_ADMIN_KEY}\"")
print("copilot -i \"I just joined Zava as a senior engineer. According to company policy, how many vacation weeks does a senior employee get?\"")

## ✅ ミッション達成

**構築したもの:**

* ✓ **File Knowledge Source**: PDF を直接アップロードでき、チャンク化と埋め込み生成を自動で行うソース。
* ✓ **Indexed Knowledge Sources**: ビルド済みの HR と health の検索インデックス用ソース。インデックススキーマと処理を完全に制御できる。
* ✓ **マルチソース Knowledge Base**: 各サブクエスチョンを自動で適切なソースへルーティングし、引用付きの回答を返す単一の KB。

➡️ [Part 2: Web 検索結果からの grounding を追加する](part2-search-mcp-kb.ipynb) に進みましょう。